In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

In [41]:
df = pd.read_csv("../data/covid_toy.csv")

df.head(10)

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No
5,84,Female,NaN,Mild,Bangalore,Yes
6,14,Male,101.0,Strong,Bangalore,No
7,20,Female,NaN,Strong,Mumbai,Yes
8,19,Female,100.0,Strong,Bangalore,No
9,64,Female,101.0,Mild,Delhi,No


In [42]:
# Here, since the fever column has missing values.. if we try to apply a column transformer with tnf1 = simple imputer 
# and tnf2 = standard scalar on fever column.. it will throw an error because we cant scale when there are missing values.

#Also, column transformer doesnt work sequentially -- tnf1 then tnf2 ..., instead it works parallely.
#So, if we want to impute and scale together in a more efficient manner using column transformer, we need to use sklearn pipelines!

#This is done in a notebook in Sklearn pieplines folder..

#In this notebook, for fever column i will only apply simple imputer

In [43]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [44]:
X = df.drop(columns=['has_covid'])
y = df['has_covid']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state=42)

In [45]:
X_train

,age,gender,fever,cough,city
55,81,Female,101.0,Mild,Mumbai
88,5,Female,100.0,Mild,Kolkata
26,19,Female,100.0,Mild,Kolkata
42,27,Male,100.0,Mild,Delhi
69,73,Female,103.0,Mild,Delhi
...,...,...,...,...,...
60,24,Female,102.0,Strong,Bangalore
71,75,Female,104.0,Strong,Delhi
14,51,Male,104.0,Mild,Bangalore
92,82,Female,102.0,Strong,Kolkata


In [46]:
#Applying column transformer --

from sklearn.compose import ColumnTransformer

transformer = ColumnTransformer(transformers=[
    ('tnf1', StandardScaler(), ['age']),
    ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
    ('tnf3', OneHotEncoder(sparse_output=False, drop='first', dtype = np.int32), ['gender', 'city']),
    ('tnf4', SimpleImputer(), ['fever'])
], remainder='passthrough')

In [47]:
arr_train_transform = transformer.fit_transform(X_train)
arr_test_transform = transformer.transform(X_test)

In [48]:
arr_train_transform

array([[ 1.56614097e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  0.00000000e+00,  1.00000000e+00,
         1.01000000e+02],
       [-1.55894504e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  1.00000000e+00,  0.00000000e+00,
         1.00000000e+02],
       [-9.83271305e-01,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  1.00000000e+00,  0.00000000e+00,
         1.00000000e+02],
       [-6.54314883e-01,  0.00000000e+00,  1.00000000e+00,
         1.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+02],
       [ 1.23718454e+00,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.03000000e+02],
       [ 1.11382589e+00,  1.00000000e+00,  1.00000000e+00,
         0.00000000e+00,  1.00000000e+00,  0.00000000e+00,
         1.03000000e+02],
       [ 2.50315277e-01,  0.00000000e+00,  0.00000000e+00,
         1.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         1.0200000

In [49]:
X_train_transformed = pd.DataFrame(arr_train_transform, columns=['age', 'cough_strong', 'gender_male', 'city_delhi', 'city_kolkata', 'city_mumbai', 'fever'])
X_test_transformed = pd.DataFrame(arr_test_transform, columns=['age', 'cough_strong', 'gender_male', 'city_delhi', 'city_kolkata', 'city_mumbai', 'fever'])

In [50]:
X_train_transformed

,age,cough_strong,gender_male,city_delhi,city_kolkata,city_mumbai,fever
0,1.566141,0.0,0.0,0.0,0.0,1.0,101.0
1,-1.558945,0.0,0.0,0.0,1.0,0.0,100.0
2,-0.983271,0.0,0.0,0.0,1.0,0.0,100.0
3,-0.654315,0.0,1.0,1.0,0.0,0.0,100.0
4,1.237185,0.0,0.0,1.0,0.0,0.0,103.0
...,...,...,...,...,...,...,...
75,-0.777674,1.0,0.0,0.0,0.0,0.0,102.0
76,1.319424,1.0,0.0,1.0,0.0,0.0,104.0
77,0.332554,0.0,1.0,0.0,0.0,0.0,104.0
78,1.607261,1.0,0.0,0.0,1.0,0.0,102.0


In [53]:
# To apply imputation and scaling together in column transformer we can use sklearn pipelines!
# Column transformers are used for preprocessing on training features, for label-encoding do it separately after applying column transformer